# Build and run PARSeq with TensorRT

Point the notebook to an exported Base, Tiny or Fast Tiny bundle. Copy the runtime cells in order into your own Python file to use the same engine builder and inference class without the training repository. You need PyTorch with CUDA, TensorRT, Pillow and NumPy.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DEPLOYMENT = ROOT / os.environ.get("PARSEQ_DEPLOYMENT", "artifacts/dummy_tiny/deployment")
IMAGE_PATH = ROOT / os.environ.get("PARSEQ_IMAGE_PATH", "dummy_data/plates/val/000.png")
# Retain compatibility with the numbered training and export notebooks.
if "PARSEQ_CONFIG" in os.environ and "PARSEQ_DEPLOYMENT" not in os.environ:
    import sys
    sys.path.insert(0, str(ROOT))
    from parseq_ocr.config import load_config
    config = load_config(os.environ["PARSEQ_CONFIG"])
    DEPLOYMENT = config["output_dir"] / "deployment"
    if "PARSEQ_IMAGE_PATH" not in os.environ:
        from parseq_ocr.data import scan_pairs
        IMAGE_PATH = Path(scan_pairs(config["dataset_root"] / "val", config["normalization"])[0]["path"])

## Copyable runtime

Image preparation and Fast Tiny's single-graph builder and runtime.

In [ ]:
from PIL import Image

def normalize(text, mode="none"):
    if mode not in (None, "none"):
        raise ValueError("Literal text is required; normalization must be none")
    return text


def raw_image_tensor(images, size):
    h, w = size
    return np.stack([np.asarray(im.convert("RGB").resize((w, h), Image.Resampling.BICUBIC),
                                dtype=np.float32).transpose(2, 0, 1) for im in images])


"""TensorRT single-graph build and optional CUDA graph replay, using PyTorch CUDA."""
import hashlib
import json
from pathlib import Path
import numpy as np
import torch
import tensorrt as trt

LOGGER = trt.Logger(trt.Logger.ERROR)


def build_single_graph(folder, max_batch=32, precision="fp16", force=False):
    folder = Path(folder)
    if max_batch < 1 or precision not in ("fp16", "fp32", "bf16"):
        raise ValueError("Invalid max_batch or precision")
    source, target = folder/"model.onnx", folder/"model.engine"
    meta = json.loads((folder/"metadata.json").read_text())
    info = {"onnx_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
            "precision_enabled": precision, "decoder": "FP32 constrained" if precision != "fp32" else "FP32",
            "max_batch": max_batch, "gpu": torch.cuda.get_device_name(), "tensorrt_version": trt.__version__}
    resize_preview = getattr(trt.PreviewFeature, "RUNTIME_ACTIVATION_RESIZE_10_10", None)
    info["activation_resize"] = resize_preview is not None
    sidecar = folder/"model.build.json"
    if target.exists() and not force:
        if not sidecar.exists() or json.loads(sidecar.read_text()) != info:
            raise ValueError("Cached engine differs from source/target. Rebuild with force=True.")
        return {"model": info}
    builder = trt.Builder(LOGGER)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, LOGGER)
    if not parser.parse(source.read_bytes()):
        raise RuntimeError("\n".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)
    if resize_preview is not None:
        config.set_preview_feature(resize_preview, True)
    if precision != "fp32":
        config.set_flag(trt.BuilderFlag.FP16 if precision == "fp16" else trt.BuilderFlag.BF16)
        config.set_flag(trt.BuilderFlag.OBEY_PRECISION_CONSTRAINTS)
        for layer in network:
            if "/encoder/" not in layer.name and layer.type != trt.LayerType.CONSTANT:
                floating = [i for i in range(layer.num_outputs) if layer.get_output(i).dtype in (trt.float32, trt.float16, trt.bfloat16)]
                if floating:
                    layer.precision = trt.float32
                    for i in floating:
                        layer.set_output_type(i, trt.float32)
    h, w = meta["height"], meta["width"]
    profile = builder.create_optimization_profile()
    profile.set_shape("pixels", (1,3,h,w), (min(8,max_batch),3,h,w), (max_batch,3,h,w))
    config.add_optimization_profile(profile)
    blob = builder.build_serialized_network(network, config)
    if blob is None:
        raise RuntimeError("Single-graph TensorRT build failed")
    target.write_bytes(blob)
    sidecar.write_text(json.dumps(info, indent=2))
    (folder/"engine_build.json").write_text(json.dumps({"model": info}, indent=2))
    return {"model": info}


class _Execution:
    def __init__(self, engine, batch, meta, capture):
        self.context = engine.create_execution_context(trt.ExecutionContextAllocationStrategy.USER_MANAGED)
        if not self.context.set_input_shape("pixels", (batch,3,meta["height"],meta["width"])):
            raise ValueError("Unsupported execution shape")
        self.workspace_bytes = self.context.update_device_memory_size_for_shapes()
        if self.workspace_bytes < 0:
            raise RuntimeError("Could not determine TensorRT activation memory")
        # Keep this allocation alive for the execution context and captured graph.
        # CUDA allocations must provide at least 256-byte alignment.
        allocated = max(256, self.workspace_bytes)
        self.workspace = torch.empty(allocated + 255, device="cuda", dtype=torch.uint8)
        address = (self.workspace.data_ptr() + 255) // 256 * 256
        self.context.set_device_memory(address, allocated)
        self.buffers = {}
        for i in range(engine.num_io_tensors):
            name = engine.get_tensor_name(i)
            dtype = torch.from_numpy(np.empty((), dtype=trt.nptype(engine.get_tensor_dtype(name)))).dtype
            shape = tuple(self.context.get_tensor_shape(name))
            self.buffers[name] = torch.zeros(shape, device="cuda", dtype=dtype)
            self.context.set_tensor_address(name, self.buffers[name].data_ptr())
        self.graph = None
        for _ in range(2):
            self.enqueue()
        torch.cuda.synchronize()
        if capture:
            self.graph = torch.cuda.CUDAGraph()
            with torch.cuda.graph(self.graph):
                self.enqueue()

    def enqueue(self):
        if not self.context.execute_async_v3(torch.cuda.current_stream().cuda_stream):
            raise RuntimeError("TensorRT enqueue failed")

    def run(self, pixels):
        self.buffers["pixels"].copy_(torch.from_numpy(pixels))
        if self.graph is None:
            self.enqueue()
        else:
            self.graph.replay()
        return self.buffers["ids"].cpu().tolist()


class SingleGraphTensorRTRecognizer:
    """One model engine; separate persistent execution/capture per used batch size.

    CUDA capture skips Python engine dispatch on replay. Input copying and text
    decoding remain outside the graph. Activation memory is allocated per used
    shape; enabling runtime activation resize at engine build avoids reserving
    maximum-batch scratch for every small batch. Captured shapes retain their
    buffers until close(). Instances are not thread safe.
    """
    def __init__(self, folder, cuda_graph=None):
        folder = Path(folder)
        self.meta = json.loads((folder/"metadata.json").read_text())
        self.runtime = trt.Runtime(LOGGER)
        self.engine = self.runtime.deserialize_cuda_engine((folder/"model.engine").read_bytes())
        if self.engine is None:
            raise RuntimeError("TensorRT deserialization failed")
        self.max_batch = self.engine.get_tensor_profile_shape("pixels", 0)[2][0]
        self.capture = self.meta.get("cuda_graph", True) if cuda_graph is None else cuda_graph
        self.executions = {}

    @torch.inference_mode()
    def infer(self, images):
        n, m = len(images), self.meta
        if not 1 <= n <= self.max_batch:
            raise ValueError(f"Batch must be 1..{self.max_batch}")
        pixels = raw_image_tensor(images, (m["height"], m["width"]))
        if n not in self.executions:
            self.executions[n] = _Execution(self.engine, n, m, self.capture)
        rows = self.executions[n].run(pixels)
        texts = []
        for ids in rows:
            if m["eos_id"] in ids:
                ids = ids[:ids.index(m["eos_id"])]
            texts.append("".join(m["vocabulary"][i] for i in ids))
        return texts

    def close(self):
        torch.cuda.synchronize()
        self.executions.clear()

Base and Tiny use split encoder and decoder graphs. The runtime detects the exported graph type.

In [ ]:
"""GPU-resident TensorRT inference; training and ONNX Runtime are not required."""
from pathlib import Path
import pathlib
import json
import hashlib
import numpy as np
import torch
import tensorrt as trt
from PIL import Image

LOGGER = trt.Logger(trt.Logger.ERROR)


def build_engines(folder, max_batch=32, precision=None, force=False):
    """Build for this GPU; refinement remains FP32 on the tested TensorRT release."""
    import resource
    resource.setrlimit(resource.RLIMIT_CORE, (0, 0))
    folder = Path(folder)
    meta = json.loads((folder/"metadata.json").read_text())
    if meta.get("graph_type") == "single_graph":
        return build_single_graph(folder, max_batch, precision or meta.get("trt_precision", "fp16"), force)
    precision = precision or "fp16"
    names = ["encoder", "decoder"]
    if not meta["decoding"]["autoregressive"]:
        names.append("parallel")
    if meta["decoding"]["refinements"]:
        names.append("refine")
    if precision not in ("fp16", "bf16", "fp32") or max_batch < 1:
        raise ValueError("Invalid precision or max_batch")
    descriptions = {}
    for name in names:
        source, target = folder/f"{name}.onnx", folder/f"{name}.engine"
        sidecar = folder/f"{name}.build.json"
        info = {"onnx_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
                "precision_enabled": "fp32" if name == "refine" else precision,
                "max_batch": max_batch, "gpu": torch.cuda.get_device_name(),
                "tensorrt_version": trt.__version__}
        if target.exists() and not force:
            if not sidecar.exists() or json.loads(sidecar.read_text()) != info:
                raise ValueError(f"Cached {name} engine differs from its source/target. Rebuild with force=True.")
            descriptions[name] = info
            continue
        builder = trt.Builder(LOGGER)
        network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
        parser = trt.OnnxParser(network, LOGGER)
        if not parser.parse(source.read_bytes()):
            raise RuntimeError("\n".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)
        if info["precision_enabled"] != "fp32":
            config.set_flag(trt.BuilderFlag.FP16 if info["precision_enabled"] == "fp16" else trt.BuilderFlag.BF16)
        profile = builder.create_optimization_profile()
        optimal = min(8, max_batch)
        if name == "encoder":
            h, w = meta["height"], meta["width"]
            profile.set_shape("pixels", (1,3,h,w), (optimal,3,h,w), (max_batch,3,h,w))
        else:
            n, d = meta["image_tokens"], meta["embedding_dim"]
            profile.set_shape("memory", (1,n,d), (optimal,n,d), (max_batch,n,d))
            profile.set_shape("tokens", (1,1), (optimal,8), (max_batch,meta["max_steps"]))
        config.add_optimization_profile(profile)
        blob = builder.build_serialized_network(network, config)
        if blob is None:
            raise RuntimeError(f"TensorRT failed to build {name}")
        target.write_bytes(blob)
        sidecar.write_text(json.dumps(info, indent=2))
        descriptions[name] = info
        print(f"Built {target.name} ({info['precision_enabled']})", flush=True)
    (folder/"engine_build.json").write_text(json.dumps(descriptions, indent=2))
    return descriptions

class Engine:
    def __init__(self,path):
        self.runtime=trt.Runtime(LOGGER)
        self.engine=self.runtime.deserialize_cuda_engine(pathlib.Path(path).read_bytes())
        if self.engine is None:raise RuntimeError('TensorRT deserialization failed')
        self.context=self.engine.create_execution_context()
        self.names=[self.engine.get_tensor_name(i) for i in range(self.engine.num_io_tensors)]
        self.outputs=[n for n in self.names if self.engine.get_tensor_mode(n)==trt.TensorIOMode.OUTPUT]

    def __call__(self,inputs):
        buffers={}
        for name,value in inputs.items():
            dtype=torch.from_numpy(np.empty((),dtype=trt.nptype(self.engine.get_tensor_dtype(name)))).dtype
            buffers[name]=value.to(dtype).contiguous()
            if not self.context.set_input_shape(name,tuple(value.shape)):
                raise RuntimeError(f'Invalid input shape {name}: {value.shape}')
        for name in self.outputs:
            shape=tuple(self.context.get_tensor_shape(name))
            dtype=torch.from_numpy(np.empty((),dtype=trt.nptype(self.engine.get_tensor_dtype(name)))).dtype
            buffers[name]=torch.empty(shape,device='cuda',dtype=dtype)
        for name,value in buffers.items():self.context.set_tensor_address(name,value.data_ptr())
        if not self.context.execute_async_v3(torch.cuda.current_stream().cuda_stream):
            raise RuntimeError('TensorRT execution failed')
        return {name:buffers[name] for name in self.outputs}

class TensorRTRecognizer:
    def __init__(self,folder):
        folder=pathlib.Path(folder);self.meta=json.loads((folder/'metadata.json').read_text())
        self.single = None
        if self.meta.get("graph_type") == "single_graph":
            self.single = SingleGraphTensorRTRecognizer(folder)
            return
        self.encoder=Engine(folder/'encoder.engine');self.decoder=Engine(folder/'decoder.engine')
        self.parallel=Engine(folder/'parallel.engine') if not self.meta['decoding']['autoregressive'] else None
        self.refine=Engine(folder/'refine.engine') if self.meta['decoding']['refinements'] else None
    @torch.inference_mode()
    def infer(self,images):
        if self.single is not None:
            return self.single.infer(images)
        m=self.meta;b=len(images);x=np.stack([np.asarray(im.resize((m['width'],m['height']),Image.Resampling.BICUBIC),dtype=np.float32).transpose(2,0,1) for im in images]);pixels=torch.from_numpy(x).cuda()/127.5-1
        memory=self.encoder({'pixels':pixels})['memory'];tokens=torch.full((b,1),m['bos_id'],device='cuda',dtype=torch.int64)
        if self.parallel:logits=self.parallel({'memory':memory,'tokens':tokens})['logits']
        else:
            emitted=[];finished=torch.zeros(b,device='cuda',dtype=torch.bool)
            for _ in range(m['max_steps']):
                z=self.decoder({'memory':memory,'tokens':tokens})['logits'];emitted.append(z);ids=z[:,0].argmax(-1);tokens=torch.cat([tokens,ids[:,None]],1);finished|=ids==m['eos_id']
                if finished.all().item():break
            logits=torch.cat(emitted,1)
        for _ in range(m['decoding']['refinements']):
            prefix=torch.cat([torch.full((b,1),m['bos_id'],device='cuda',dtype=torch.int64),logits[:,:-1].argmax(-1)],1)
            logits=self.refine({'memory':memory,'tokens':prefix})['logits']
        if not torch.isfinite(logits).all().item():raise RuntimeError('Nonfinite TensorRT logits')
        texts=[]
        for row in logits.argmax(-1).cpu().tolist():
            if m['eos_id'] in row:row=row[:row.index(m['eos_id'])]
            texts.append(''.join(m['vocabulary'][i] for i in row))
        return texts

    def close(self):
        if self.single is not None:
            self.single.close()

Use the integration class to build missing engines and read a crop.

In [ ]:
class PARSeqTensorRT(TensorRTRecognizer):
    def __init__(self, deployment_dir, *, max_batch=32, force_rebuild=False):
        build_engines(deployment_dir, max_batch=max_batch, force=force_rebuild)
        super().__init__(deployment_dir)

    def predict(self, image):
        if isinstance(image, (str, Path)):
            with Image.open(image) as source:
                return self.infer([source])[0]
        return self.infer([image])[0]

## Build and read an image

In [ ]:
MAX_BATCH = 32
reader = PARSeqTensorRT(DEPLOYMENT, max_batch=MAX_BATCH)
try:
    with Image.open(IMAGE_PATH) as image:
        display(image.copy())
        prediction = reader.predict(image)
    print(repr(prediction))
finally:
    reader.close()